# Gemeindedataset

In [1]:
from pathlib import Path
import pandas as pd


# ------------------------------------------------------------
# 1. Dateien und Bezugsjahre
# ------------------------------------------------------------

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "scripts" else Path.cwd()
BFS_DIR = PROJECT_DIR / "data_processed/bfs_data/mapexplorer/csv"

POPULATION_FILE = BFS_DIR / ("staendige_wohnbevoelkerung__ch_01_02_01a__staendigepop.csv")
DENSITY_FILE = BFS_DIR / ("einwohner_innen_pro_km2_produktivflaeche__ch_01_02_01a__bevoelkerungsdichtevf.csv")
HOUSEHOLD_FILE = BFS_DIR / ("durchschnittliche_haushaltsgroesse_anzahl_personen__ch_01_07a__durchschnittlichehau.csv")
VACANCY_FILE = BFS_DIR / ("anteil_leer_stehender_wohnungen_am_gesamtwohnungsbestand__ch_09_03b__leerwohnungsziffer.csv")
AGE_FILE = BFS_DIR / ("anteil_der_personen_ab_65_jahren_an_der_staendigen_wohnbevoelkerung__ch_01_02_01a__anteilpop65.csv")
NEW_DWELLINGS_FILE = BFS_DIR / ("neu_erstellte_wohnungen_pro_1_000_einwohner__ch_09_04__neuerstelltewohnungenpro.csv")
SINGLE_FAMILY_FILE = BFS_DIR / ("anteil_einfamilienhaeuser_am_gebaeudebestand_mit_wohnnutzung__ch_09_03a__anteileinfamhaus.csv")
FOREIGN_POPULATION_FILE = BFS_DIR / ("anteil_der_staendigen_auslaendischen_wohnbevoelkerung_an_der_staendigen_wohnbevoelkerung__ch_01_02_01a__anteilauslpop.csv")
MIGRATION_FILE = BFS_DIR / ("gesamtwanderungssaldo_pro_1_000_einwohner_innen__ch_01_04_05a__wanderungssaldopro.csv")

EMPLOYMENT_FILES = [
    BFS_DIR / "beschaeftigte_im_1_wirtschaftssektor__ch_06_02_01a__bsws1.csv",
    BFS_DIR / "beschaeftigte_im_2_wirtschaftssektor__ch_06_02_01a__bsws2.csv",
    BFS_DIR / "beschaeftigte_im_3_wirtschaftssektor__ch_06_02_01a__bsws3.csv",
]

SOCIAL_FILE = PROJECT_DIR / "demographic_data/sozialhilfequote.csv"
INCOME_FILE = (PROJECT_DIR / "tax_data/statistik-dbst-np-kennzahlen-ohne-2022.csv")
OUTPUT_FILE = (PROJECT_DIR / "data_processed/gemeinde_indikatoren_zentralschweiz.csv")

POPULATION_FROM_YEAR = 2024
POPULATION_TO_YEAR = 2025
DENSITY_YEAR = 2025
HOUSEHOLD_YEAR = 2025
VACANCY_YEAR = 2025
SOCIAL_YEAR = 2024
SOCIAL_MUNICIPALITY_STAND = "01.01.2024"
INCOME_YEAR = 2022

# Bisher verwendete Einkommensmetrik.
# Für das steuerbare Median-Äquivalenzeinkommen: "median_steinka".
INCOME_METRIC = "median_steink"

# Bei den sechs Ergänzungen jüngsten verfügbaren Jahrgang bis 2025 wählen.
MAX_YEAR = 2025


# Hilfsfunktionen
def load_bfs(path):
    """Lädt eine Map-Explorer-CSV und vereinheitlicht numerische Spalten."""
    data = pd.read_csv(
        path, sep=None, engine="python", encoding="utf-8-sig"
    )

    required = {"gemeinde_bfs", "jahr", "wert", "gemeindestand"}
    if not required.issubset(data.columns):
        raise ValueError(
            f"{path.name}: Benötigt werden {required}. "
            f"Vorhanden: {data.columns.tolist()}"
        )

    data["gemeinde_bfs"] = pd.to_numeric(data["gemeinde_bfs"], errors="raise").astype("Int64")
    data["jahr"] = pd.to_numeric(data["jahr"], errors="raise").astype("Int64")
    data["wert"] = pd.to_numeric(data["wert"], errors="coerce")

    return data.dropna(subset=["gemeinde_bfs", "jahr"]).copy()


def select_year(data, year):
    """Wählt einen Jahrgang mit genau einer Zeile je Gemeinde."""
    selected = data.loc[data["jahr"].eq(year)].copy()

    if selected.empty:
        raise ValueError(f"Keine Daten für das Jahr {year} vorhanden.")

    if selected["gemeinde_bfs"].duplicated().any():
        raise ValueError(
            f"Mehrere Zeilen je Gemeinde im Jahr {year}. "
            "Bitte Raumebenen und Gemeindestände prüfen."
        )

    return selected


def available_years(data, municipality_ids, max_year=MAX_YEAR):
    """Liefert Jahre mit verfügbaren Werten für die ausgewählten Gemeinden."""
    mask = (
        data["jahr"].le(max_year)
        & data["wert"].notna()
        & data["gemeinde_bfs"].isin(municipality_ids)
    )
    return set(data.loc[mask, "jahr"].astype(int))


def merge_indicator(base, data, year, column_name):
    """Ergänzt Werte, Bezugsjahr und Gemeindestand eines Indikators."""
    selected = select_year(data, year)

    values = selected[
        ["gemeinde_bfs", "wert", "gemeindestand"]
    ].rename(
        columns={
            "wert": column_name,
            "gemeindestand": f"{column_name}_gemeindestand",
        }
    )
    values[f"{column_name}_year"] = year

    return base.merge(
        values,
        on="gemeinde_bfs",
        how="left",
        validate="one_to_one",
    )


# Alle Gemeinden der Zentralschweiz
population = load_bfs(POPULATION_FILE)
population_current = select_year(population, POPULATION_TO_YEAR)

df_gemeinden = population_current[["gemeinde_bfs", "gemeinde_name", "gemeindestand"]
    ].rename(columns={"gemeindestand": "gemeindeverzeichnis_stand"}).copy()

canton_ranges = {
    "LU": (1000, 1199),
    "UR": (1200, 1299),
    "SZ": (1300, 1399),
    "OW": (1400, 1499),
    "NW": (1500, 1599),
    "ZG": (1700, 1799),
}

df_gemeinden["kanton_geo"] = pd.Series(pd.NA, index=df_gemeinden.index, dtype="string")

for canton, (lower, upper) in canton_ranges.items():
    mask = df_gemeinden["gemeinde_bfs"].between(lower, upper)
    df_gemeinden.loc[mask, "kanton_geo"] = canton

df_gemeinden = (df_gemeinden.dropna(subset=["kanton_geo"]).reset_index(drop=True))

municipality_ids = df_gemeinden["gemeinde_bfs"]
year_summary = []


# Bevölkerungswachstum 2024 → 2025
previous = select_year(population, POPULATION_FROM_YEAR)
current = population_current.loc[population_current["gemeinde_bfs"].isin(municipality_ids)]

growth = current[
    ["gemeinde_bfs", "wert", "gemeindestand"]
].merge(
    previous[["gemeinde_bfs", "wert", "gemeindestand"]],
    on="gemeinde_bfs",
    how="left",
    suffixes=("_to", "_from"),
    validate="one_to_one",
)

# Datumsangaben vor dem Vergleich vereinheitlichen.
stand_to = pd.to_datetime(growth["gemeindestand_to"], format="%d.%m.%Y", errors="raise")
stand_from = pd.to_datetime(growth["gemeindestand_from"], format="%d.%m.%Y", errors="raise")

different_stands = (
    stand_to.notna()
    & stand_from.notna()
    & stand_to.ne(stand_from)
)

if different_stands.any():
    raise ValueError(
        "Unterschiedliche Gemeindestände beim Bevölkerungswachstum."
    )

comparable_stands = stand_to.notna() & stand_from.notna()

growth["population_growth_pct"] = (
    growth["wert_to"]
    .div(growth["wert_from"].where(growth["wert_from"].gt(0)))
    .sub(1)
    .mul(100)
    .where(comparable_stands)
)

growth["population_growth_from_year"] = POPULATION_FROM_YEAR
growth["population_growth_to_year"] = POPULATION_TO_YEAR
growth["population_growth_gemeindestand"] = (growth["gemeindestand_to"].where(comparable_stands))

df_gemeinden = df_gemeinden.merge(
    growth[
        [
            "gemeinde_bfs",
            "population_growth_pct",
            "population_growth_from_year",
            "population_growth_to_year",
            "population_growth_gemeindestand",
        ]
    ],
    on="gemeinde_bfs",
    how="left",
    validate="one_to_one",
)

year_summary.append({"variable": "population_growth_pct","bezugsjahr": f"{POPULATION_FROM_YEAR} → {POPULATION_TO_YEAR}",})

# Dichte, Haushaltsgrösse und Leerstand
fixed_indicators = [
    (DENSITY_FILE, DENSITY_YEAR, "population_density_productive"),
    (HOUSEHOLD_FILE, HOUSEHOLD_YEAR, "household_size_mean"),
    (VACANCY_FILE, VACANCY_YEAR, "vacancy_rate_pct"),
]

for path, year, column_name in fixed_indicators:
    df_gemeinden = merge_indicator(
        df_gemeinden, load_bfs(path), year, column_name
    )

    year_summary.append({
        "variable": column_name,
        "bezugsjahr": str(year),
    })


# Sozialhilfequote
social = pd.read_csv(
    SOCIAL_FILE,
    sep=";",
    skiprows=2,
    encoding="utf-8-sig",
    na_values=["N/A - Statistikgeheimnis"],
)

social = social[
    [
        "Identifikator",
        f"Sozialhilfequote {SOCIAL_YEAR}",
        f"Sozialhilfequote {SOCIAL_YEAR} - Schätzung (*=ja)",
    ]
].rename(
    columns={
        "Identifikator": "gemeinde_bfs",
        f"Sozialhilfequote {SOCIAL_YEAR}": "social_assistance_rate_pct",
        f"Sozialhilfequote {SOCIAL_YEAR} - Schätzung (*=ja)":
            "social_assistance_estimated",
    }
).copy()

social["gemeinde_bfs"] = pd.to_numeric(social["gemeinde_bfs"], errors="raise").astype("Int64")
social["social_assistance_rate_pct"] = pd.to_numeric(social["social_assistance_rate_pct"], errors="raise")

social["social_assistance_estimated"] = (
    social["social_assistance_estimated"]
    .fillna("")
    .str.strip()
    .eq("*")
    .astype("boolean")
)

social["social_assistance_rate_pct_year"] = SOCIAL_YEAR
social["social_assistance_rate_pct_gemeindestand"] = (SOCIAL_MUNICIPALITY_STAND)

df_gemeinden = df_gemeinden.merge(
    social.dropna(subset=["gemeinde_bfs"]),
    on="gemeinde_bfs",
    how="left",
    validate="one_to_one",
)

year_summary.append({
    "variable": "social_assistance_rate_pct",
    "bezugsjahr": str(SOCIAL_YEAR),
})


# Einkommensmedian aus der Steuerdatei
tax = pd.read_csv(
    INCOME_FILE,
    sep=";",
    encoding="utf-8-sig",
    dtype="string",
)

tax.columns = [
    " ".join(str(column).replace("\ufeff", "").split())
    for column in tax.columns
]

municipality_columns = [
    column for column in tax.columns
    if column.startswith("Gemeindenummer")
]
unit_columns = [
    column for column in tax.columns
    if column.startswith("Einheit ID")
]

if len(municipality_columns) != 1 or len(unit_columns) != 1:
    raise ValueError(
        f"Steuerspalten nicht eindeutig erkannt: {tax.columns.tolist()}"
    )

income = tax.loc[
    tax[unit_columns[0]].str.strip().eq("T"),
    [municipality_columns[0], INCOME_METRIC],
].rename(
    columns={
        municipality_columns[0]: "gemeinde_bfs",
        INCOME_METRIC: "income_median_chf",
    }
).copy()

income["gemeinde_bfs"] = pd.to_numeric(income["gemeinde_bfs"], errors="raise").astype("Int64")
income["income_median_chf"] = pd.to_numeric(income["income_median_chf"].str.replace(r"['’\s]", "", regex=True),errors="coerce",)
income["income_year"] = INCOME_YEAR
income["income_metric"] = INCOME_METRIC

df_gemeinden = df_gemeinden.merge(
    income.dropna(subset=["gemeinde_bfs"]),
    on="gemeinde_bfs",
    how="left",
    validate="one_to_one",
)

year_summary.append({
    "variable": "income_median_chf",
    "bezugsjahr": str(INCOME_YEAR),
})

# Alter, Neubau, Gebäudestruktur, Ausländeranteil und Wanderung
additional_indicators = [
    (AGE_FILE, "population_age65plus_pct"),
    (NEW_DWELLINGS_FILE, "new_dwellings_per_1000_residents"),
    (SINGLE_FAMILY_FILE, "single_family_buildings_pct"),
    (FOREIGN_POPULATION_FILE, "foreign_population_pct"),
    (MIGRATION_FILE, "net_migration_per_1000_residents"),
]

for path, column_name in additional_indicators:
    data = load_bfs(path)
    years = available_years(data, municipality_ids)

    if not years:
        raise ValueError(f"Keine verwendbaren Jahre in {path.name}.")

    year = max(years)

    df_gemeinden = merge_indicator(
        df_gemeinden, data, year, column_name
    )

    year_summary.append({
        "variable": column_name,
        "bezugsjahr": str(year),
    })

# Beschäftigte je 100 Einwohner
sector_data = [load_bfs(path) for path in EMPLOYMENT_FILES]
common_years = available_years(population, municipality_ids)

for data in sector_data:
    common_years = common_years.intersection(
        available_years(data, municipality_ids)
    )

if not common_years:
    raise ValueError(
        "Kein gemeinsames Jahr für Beschäftigte und Bevölkerung vorhanden."
    )

employment_year = max(common_years)

employment = select_year(population, employment_year)[
    ["gemeinde_bfs", "wert", "gemeindestand"]
].rename(
    columns={
        "wert": "population",
        "gemeindestand": "employment_population_gemeindestand",
    }
)

employment = employment.loc[
    employment["gemeinde_bfs"].isin(municipality_ids)
].copy()

sector_columns = []

for sector, data in enumerate(sector_data, start=1):
    values = select_year(data, employment_year)[
        ["gemeinde_bfs", "wert", "gemeindestand"]
    ].copy()

    # Dieser Abgleich ist für die beobachteten Quelldaten definiert.
    source_stands = set(values["gemeindestand"].dropna())
    if (
        source_stands != {"01.01.2024"}
        or values["gemeindestand"].isna().any()
    ):
        raise ValueError(
            f"Sektor {sector}: Unerwarteter Gemeindestand. "
            "Die Zuordnung muss erneut geprüft werden."
        )

    # Root im Gemeindestand 2025 umfasst das frühere Honau.
    if not {1057, 1065}.issubset(set(values["gemeinde_bfs"])):
        raise ValueError(
            f"Sektor {sector}: Honau oder Root fehlt in den Quelldaten."
        )

    values["gemeinde_bfs"] = values["gemeinde_bfs"].replace({1057: 1065})

    # Nur bei vollständig vorhandenen Ausgangswerten summieren.
    # Geheimgehaltene oder fehlende Werte bleiben fehlend.
    values = (
        values.groupby("gemeinde_bfs", as_index=False)["wert"]
        .agg(lambda series: series.sum(min_count=len(series)))
    )

    column_name = f"sector_{sector}"
    values = values.rename(columns={"wert": column_name})

    employment = employment.merge(
        values,
        on="gemeinde_bfs",
        how="left",
        validate="one_to_one",
    )
    sector_columns.append(column_name)

employment_total = employment[sector_columns].sum(axis=1, min_count=3)

employment["employment_per_100_residents"] = (
    employment_total
    .div(employment["population"].where(employment["population"].gt(0)))
    .mul(100)
)

employment["employment_per_100_residents_year"] = employment_year
employment["employment_source_gemeindestand"] = "01.01.2024"
employment["employment_boundary_adjustment"] = (
    employment["gemeinde_bfs"]
    .eq(1065)
    .map({
        True: "Beschäftigte von Honau und Root zusammengefasst",
        False: "Keine Fusionsaggregation vorgenommen",
    })
)

df_gemeinden = df_gemeinden.merge(
    employment[
        [
            "gemeinde_bfs",
            "employment_per_100_residents",
            "employment_per_100_residents_year",
            "employment_source_gemeindestand",
            "employment_population_gemeindestand",
            "employment_boundary_adjustment",
        ]
    ],
    on="gemeinde_bfs",
    how="left",
    validate="one_to_one",
)

year_summary.append({
    "variable": "employment_per_100_residents",
    "bezugsjahr": str(employment_year),
})


# Ergebnis und Vollständigkeit

features = [
    "income_median_chf",
    "population_growth_pct",
    "population_density_productive",
    "household_size_mean",
    "vacancy_rate_pct",
    "social_assistance_rate_pct",
    "population_age65plus_pct",
    "employment_per_100_residents",
    "new_dwellings_per_1000_residents",
    "single_family_buildings_pct",
    "foreign_population_pct",
    "net_migration_per_1000_residents",
]

first_columns = [
    "gemeinde_bfs",
    "gemeinde_name",
    "kanton_geo",
    *features,
]

metadata_columns = [
    column for column in df_gemeinden.columns
    if column not in first_columns
]

df_gemeinden = (
    df_gemeinden[first_columns + metadata_columns]
    .sort_values(["kanton_geo", "gemeinde_name"])
    .reset_index(drop=True)
)

print(f"{len(df_gemeinden)} Gemeinden")
display(df_gemeinden)

print("Verwendete Bezugsjahre:")
display(pd.DataFrame(year_summary))

print("Fehlende Werte je Variable:")
display(
    df_gemeinden[features]
    .isna()
    .sum()
    .rename("Gemeinden mit fehlendem Wert")
    .to_frame()
)

# Die Gemeindestände bleiben dokumentiert.
# Gemeindegrenzen werden durch diesen Code nicht harmonisiert.

# CSV speichern
OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)

df_gemeinden.to_csv(
    OUTPUT_FILE,
    index=False,
    encoding="utf-8-sig",
)

print(f"Gespeichert: {OUTPUT_FILE}")

157 Gemeinden


,gemeinde_bfs,gemeinde_name,kanton_geo,income_median_chf,population_growth_pct,population_density_productive,household_size_mean,vacancy_rate_pct,social_assistance_rate_pct,population_age65plus_pct,...,single_family_buildings_pct_gemeindestand,single_family_buildings_pct_year,foreign_population_pct_gemeindestand,foreign_population_pct_year,net_migration_per_1000_residents_gemeindestand,net_migration_per_1000_residents_year,employment_per_100_residents_year,employment_source_gemeindestand,employment_population_gemeindestand,employment_boundary_adjustment
0,1051,Adligenswil,LU,58000,1.548870,830.3,2.37,0.64,1.1,25.1,...,04.06.2025,2025,04.06.2025,2025,04.06.2025,2025,2024,01.01.2024,04.06.2025,Keine Fusionsaggregation vorgenommen
1,1021,Aesch (LU),LU,55900,5.024088,343.7,2.26,0.94,1.0,18.5,...,04.06.2025,2025,04.06.2025,2025,04.06.2025,2025,2024,01.01.2024,04.06.2025,Keine Fusionsaggregation vorgenommen
2,1121,Alberswil,LU,49950,0.816327,212.3,2.25,0.58,0.9,13.9,...,04.06.2025,2025,04.06.2025,2025,04.06.2025,2025,2024,01.01.2024,04.06.2025,Keine Fusionsaggregation vorgenommen
3,1122,Altbüron,LU,48650,-0.098619,150.3,2.36,0.84,0.8,18.0,...,04.06.2025,2025,04.06.2025,2025,04.06.2025,2025,2024,01.01.2024,04.06.2025,Keine Fusionsaggregation vorgenommen
4,1123,Altishofen,LU,51500,-0.826446,143.1,2.54,0.24,0.8,13.7,...,04.06.2025,2025,04.06.2025,2025,04.06.2025,2025,2024,01.01.2024,04.06.2025,Keine Fusionsaggregation vorgenommen
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
152,1707,Risch,ZG,64900,1.336303,811.9,2.32,0.46,1.4,16.0,...,04.06.2025,2025,04.06.2025,2025,04.06.2025,2025,2024,01.01.2024,04.06.2025,Keine Fusionsaggregation vorgenommen
153,1708,Steinhausen,ZG,62600,0.480815,2089.8,2.30,0.21,1.0,20.2,...,04.06.2025,2025,04.06.2025,2025,04.06.2025,2025,2024,01.01.2024,04.06.2025,Keine Fusionsaggregation vorgenommen
154,1709,Unterägeri,ZG,59700,0.337019,380.5,2.26,0.35,1.2,21.2,...,04.06.2025,2025,04.06.2025,2025,04.06.2025,2025,2024,01.01.2024,04.06.2025,Keine Fusionsaggregation vorgenommen
155,1710,Walchwil,ZG,74600,0.450563,308.7,2.29,0.90,0.7,22.2,...,04.06.2025,2025,04.06.2025,2025,04.06.2025,2025,2024,01.01.2024,04.06.2025,Keine Fusionsaggregation vorgenommen


Verwendete Bezugsjahre:


,variable,bezugsjahr
0,population_growth_pct,2024 → 2025
1,population_density_productive,2025
2,household_size_mean,2025
3,vacancy_rate_pct,2025
4,social_assistance_rate_pct,2024
5,income_median_chf,2022
6,population_age65plus_pct,2025
7,new_dwellings_per_1000_residents,2024
8,single_family_buildings_pct,2025
9,foreign_population_pct,2025


Fehlende Werte je Variable:


,Gemeinden mit fehlendem Wert
income_median_chf,0
population_growth_pct,0
population_density_productive,0
household_size_mean,0
vacancy_rate_pct,0
social_assistance_rate_pct,18
population_age65plus_pct,0
employment_per_100_residents,0
new_dwellings_per_1000_residents,0
single_family_buildings_pct,0


Gespeichert: /Users/enrue/Developer/jupyter-setup/cip_projekt/data_processed/gemeinde_indikatoren_zentralschweiz.csv
